# Module 5: ML Architectures

Different data types require different architectures:
- **Tabular data** -> MLPs, tree-based models
- **Images** -> CNNs (local spatial patterns)
- **Sequences (text, time series)** -> RNNs/LSTMs (temporal dependencies)
- **Everything** -> Transformers (attention-based, the current dominant paradigm)

This notebook implements a CNN for image classification and a bidirectional LSTM for sentiment analysis.

## Part 1: CNN for Image Classification

**Why CNNs for images?** A fully connected network treating each pixel independently ignores spatial structure. CNNs use convolutions — small sliding filters that detect local patterns (edges, textures, shapes) and build up to complex features through layers.

**Architecture choices:**
- **Conv -> BatchNorm -> ReLU -> MaxPool** is the standard block
- **Two conv blocks** (32 filters -> 64 filters): first block learns edges/textures, second learns patterns/shapes
- **MaxPool(2)** halves spatial dimensions, creating translation invariance

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
    def forward(self, x):
        return self.block(x)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),   # 28x28 -> 14x14
            ConvBlock(32, 64),  # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = SimpleCNN()
total_params = sum(p.numel() for p in model.parameters())
print(f"CNN parameters: {total_params:,}")
print(f"\n{model}")

## Part 2: Bidirectional LSTM for Sentiment Analysis

**Why LSTMs for text?** Text is sequential — the meaning of a word depends on surrounding words. LSTMs maintain a "memory cell" that can selectively remember or forget information across long sequences, solving the vanishing gradient problem that plagues vanilla RNNs.

**Why bidirectional?** Reading "I didn't like the movie" forward, you don't know the sentiment until you hit "didn't." Bidirectional LSTM reads both forward and backward, capturing context from both directions. We concatenate both hidden states for the final representation.

In [ ]:
import numpy as np

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

model = SentimentLSTM(vocab_size=5000, embed_dim=128, hidden_dim=64, output_dim=2)
total_params = sum(p.numel() for p in model.parameters())
print(f"LSTM parameters: {total_params:,}")
print(f"\nEmbedding: 5000 words -> 128 dimensions")
print(f"LSTM: 128 -> 64 hidden (bidirectional = 128 total)")
print(f"Classifier: 128 -> 2 classes")

## Architecture Selection Guide

| Data Type | First Choice | Why | Alternative |
|-----------|-------------|-----|------------|
| Tabular | XGBoost | Handles mixed types, feature interactions | MLP |
| Images | CNN (ResNet, EfficientNet) | Spatial hierarchy, translation invariance | ViT (with enough data) |
| Text | Transformer (BERT, GPT) | Long-range dependencies, parallelizable | BiLSTM (smaller data) |
| Time Series | LSTM / Transformer | Temporal dependencies | 1D CNN |
| Graph | GNN (GAT, GCN) | Relational structure | Node2Vec + MLP |

**The trend:** Transformers are eating everything. Vision Transformers (ViT) compete with CNNs on images. But for tabular data, tree-based models still win.